In [1]:
!python -V

Python 3.9.25


In [1]:
import os

print(os.getcwd())
print(os.path.abspath("mlflow.db"))

/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking
/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking/mlflow.db


# Q3. Train a model with autolog

In [3]:
import os
import pickle
import click
import mlflow
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("randomforest")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
def run_train(data_path: str):
    mlflow.sklearn.autolog()
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    with mlflow.start_run():

        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))


if __name__ == '__main__':
    run_train.callback(data_path="./output")